In [ ]:
import sys
sys.path.insert(0, '../..')

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import edunn as nn
from edunn import utils

In [ ]:
np.set_printoptions(threshold=sys.maxsize)

# Gaussian Error Linear Unit

GELU is an activation function used in the hidden layers of neural networks. It is used to introduce non-linearity into neural network models and helps decide which neurons should be activated.

GELU combines the advantages of other popular activation functions such as ReLU and ELU. It provides smooth saturation for both positive and negative values, which can help mitigate the vanishing gradient problem.

## Forward Method

There are several ways to express this function; one of them uses the cumulative distribution function (CDF) and the [error function](https://en.wikipedia.org/wiki/Error_function):

$$ \text{GELU} = x \cdot \text{CDF}(x) = x \cdot \frac{1+\text{erf}\big(\frac{x}{\sqrt{2}}\big)}{2} $$

This works because the `CDF` can be expressed in terms of the `erf`, scaling the linear function `x` by the `CDF`. This version of GELU is the exact one and is the one used by default by the PyTorch deep learning library.

<center>
<img src="img/gelu.png" width="25%" style="display:inline-block">
</center>

> NOTE: the `erf` function is currently not implemented in `numpy`; you can take the time to implement it numerically or use the implementation provided by `from scipy.special import erf`.

## Backward Method

Computing it is simpler than it looks, because here we will not use the expression that depends on the `erf`, but the one that depends explicitly on the `CDF`.

$$
\begin{aligned}
(\text{x} \cdot \text{CDF}(\text{x}))' &= \text{x}' \cdot \text{CDF}(\text{x}) + \text{x} \cdot \text{CDF}'(\text{x}) \\
&= \text{CDF}(\text{x}) + \text{x} \cdot \text{PDF}(\text{x})
\end{aligned}
$$

This follows from the fundamental theorem of calculus, where by definition:

- Differentiating the CDF gives you the PDF: $\text{PDF}(x) = \frac{d}{dx} \text{CDF}(x)$
- Integrating the PDF gives you the CDF: $\text{CDF}(x) = \int_{-\infty}^{x} \text{PDF}(t) dt$

The `PDF` we will use is that of the normal or Gaussian distribution. Again, `numpy` does not provide a way to compute it over the values of an input vector, so you will need to implement the `normal_pdf` function in the `activations.py` file.

This is the formula for the probability density function (PDF) of a normal distribution:

$$
f(x, \mu=0, \sigma=1) = \frac{1}{\sqrt{2\pi\sigma^2}} e^{-\frac{(x-\mu)^2}{2\sigma^2}}
$$

where $x$ is the random variable, $\mu$ is the mean of the distribution and $\sigma$ is the standard deviation of the distribution.

<!-- The PDF (Probability Density Function) gives the probability that a random variable takes a specific value, while the CDF (Cumulative Distribution Function) gives the probability that a random variable takes a value less than or equal to a given value. -->

<!-- 1. **PDF (Probability Density Function)**: A function that describes the relative probability that a random variable takes a given value. The probability that the random variable falls within a particular range is given by the area under the graph of the density function over that interval.

2. **CDF (Cumulative Distribution Function)**: A function that indicates the probability that a random variable is less than or equal to a given value. It is the integral of the PDF up to that value.

- The PDF can take values greater than 1, while the CDF can never take values greater than 1. -->

In [ ]:
np.random.seed(123)

din=10
batch_size=2

x = np.random.rand(batch_size,din)

layer=nn.GELU()

In [ ]:
x

In [ ]:
y = layer.forward(x)
y, y.shape

In [ ]:
# plot values
nn.plot.plot_activation_function(nn.GELU())

In [ ]:
# Define the output gradient
g = np.random.rand(*y.shape)

# Propagate the gradient backward through the convolution
layer_grad = layer.backward(g)
layer_grad

In [ ]:
import torch
import torch.nn as tnn

x = torch.from_numpy(x).to(torch.float)

# We define the GELU layer
gelu = tnn.GELU()

x.requires_grad = True

y_torch = gelu(x)
y_torch

In [ ]:
utils.check_same(y_torch.detach().numpy(),y,tol=1e-5)

In [ ]:
# Define the output gradient
g = torch.from_numpy(g).to(torch.double)

# Propagate the gradient backward
x.grad = None  # We clear the existing gradients
y_torch.backward(g)

# Print the gradient of the input image
print("Input gradient (δE/δx):")
print(x.grad, x.grad.shape)

In [ ]:
utils.check_same(x.grad.numpy(),layer_grad[0],tol=1e-5)

In [ ]:
from edunn.utils import check_gradient


# Number of random examples and their size, used to generate
# samples of x and verify the derivatives
samples = 100
input_shape=(5,2)

# Check the derivatives of a GELU function
layer=nn.GELU()
check_gradient.common_layer(layer,input_shape,samples=samples)

nn.plot.plot_activation_function(nn.GELU(),backward=True)